# Kafka Consumer — Data Engineering Course

This notebook covers two consumer examples:
- **Example 1** — Basic consumer: read messages from a topic and print metadata
- **Example 2** — Advanced consumer: consume messages and write them to a local sink file (File → Kafka → File pipeline)

> Run the **Producer notebook** first to have messages available in the topics.

---
## Example 1 — Basic Consumer
Connect to a topic and print every incoming message along with its metadata.

In [ ]:
from kafka import KafkaConsumer
from datetime import datetime

# ----- Configuration -----
TOPIC   = 'kafka-tst-07'
BROKERS = 'course-kafka:9092'

In [ ]:
# Create a KafkaConsumer instance.
# KafkaConsumer is a Python GENERATOR — iterating over it yields one message at a time.
# It will BLOCK and wait for new messages when the topic is empty.
consumer = KafkaConsumer(TOPIC, bootstrap_servers=BROKERS)

# Note: without a group_id, each consumer instance gets ALL messages independently.
# With a group_id, Kafka coordinates partition assignment so each message
# is processed by exactly ONE consumer in the group (load balancing).

In [ ]:
# Each `message` object has several useful fields:
#   message.value     — the raw bytes payload
#   message.key       — optional routing key (bytes or None)
#   message.offset    — position of this message within its partition (monotonically increasing)
#   message.partition — which partition this message came from
#   message.timestamp — Unix timestamp in milliseconds (set by the producer)

for message in consumer:
    # Convert the Unix ms timestamp to a human-readable datetime
    dt_object = datetime.fromtimestamp(message.timestamp / 1000)

    print('-----------------------------')
    print('Value  :', message.value.decode('utf-8'))
    print('Offset :', message.offset)    # Useful for debugging / replay
    print('Time   :', dt_object)

---
## Example 2 — File-to-File Consumer

Full pipeline: **Log file → Producer → Kafka → Consumer → Sink file**

The consumer reads every message from the topic and writes it to a destination file,
demonstrating how Kafka decouples the data source from the data sink.

In [ ]:
from kafka import KafkaConsumer

# ----- Configuration -----
TOPIC2       = 'kafka-tst-02'
BROKERS2     = ['course-kafka:9092']
TARGET_FILE  = '/home/developer/kafka/sinkFiles/tarFile.log'

In [ ]:
# group_id: assigns this consumer to a consumer group named 'File2File'.
# All consumers sharing the same group_id split the partitions among themselves —
# no message is processed twice within the group.

# auto_commit_interval_ms: how often (in ms) the consumer automatically commits
# its current offset back to Kafka. Committing tells Kafka:
# "I have successfully processed up to this point — don't resend on restart."
consumer2 = KafkaConsumer(
    TOPIC2,
    group_id                 = 'File2File',
    bootstrap_servers        = BROKERS2,
    auto_commit_interval_ms  = 1000    # commit offset every 1 second
)

In [ ]:
# Open the sink file once and keep it open for the lifetime of the consumer.
# This is more efficient than opening/closing on every message.
with open(TARGET_FILE, 'w') as f:
    for message in consumer2:
        print('Received:', message.value)

        # Write the raw bytes representation to disk.
        # In a real pipeline you would decode + parse (e.g. json.loads) before writing.
        f.write(format(message.value) + '\n')

        # flush() forces the OS to write the in-memory file buffer to disk immediately.
        # Without this, data might sit in the buffer and be lost if the process crashes.
        f.flush()

---
### Verify — Inspect the Sink File
Run this cell after the consumer has processed some messages to confirm the data landed correctly.

In [ ]:
TARGET_FILE = '/home/developer/kafka/sinkFiles/tarFile.log'

# Simply read and print the entire sink file to verify the pipeline worked end-to-end.
with open(TARGET_FILE) as f:
    content = f.read()

print(content if content else '(file is empty — make sure the producer and consumer have run)')